# Numerical solutions — Python lab
Session 6 · 5CKI2MCP

**Work through the numbered TO DOs.** Replace `None` with a formula; change the marked numbers. Run cells from top to bottom with **Shift + Enter**. Leave the plotting code alone.

`x[n]` means $x_n$; $hx_n$ is `h * x[n]`; `/` means division; `**2` means squared. An unfinished formula produces a reminder. Double-click an **Answer** cell to write in it.


In [ ]:
# SETUP — run once; nothing to edit.
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11, "axes.grid": True, "grid.alpha": 0.25})


## 1. Complete the two Euler methods

$$
\frac{dx}{dt}=-x, \qquad x(0)=1, \qquad x(t)=e^{-t}, \qquad h=T/N.
$$

Use these updates to fill **TO DO 1.1–1.2**:

$$
\text{Explicit: }x_{n+1}=(1-h)x_n, \qquad
\text{Implicit: }x_{n+1}=\frac{x_n}{1+h}.
$$


In [ ]:
def euler_decay(T, N):
    h = T / N
    time = np.linspace(0, T, N + 1)
    explicit = np.zeros(N + 1)
    implicit = np.zeros(N + 1)
    explicit[0] = 1.0
    implicit[0] = 1.0

    for n in range(N):
        # TO DO 1.1: replace None using explicit[n] and h.
        next_explicit = None

        # TO DO 1.2: replace None using implicit[n] and h.
        next_implicit = None

        # Supplied check and storage — do not edit.
        if next_explicit is None or next_implicit is None:
            raise ValueError("Complete TO DO 1.1 and 1.2, then rerun both cells in Section 1.")
        explicit[n + 1] = next_explicit
        implicit[n + 1] = next_implicit

    return time, explicit, implicit


In [ ]:
T = 4.0
# TO DO 1.3: run with N = 8, then 16, then 32. Keep T fixed.
N = 8

# PLOTTING — supplied; nothing to edit below.
time, explicit, implicit = euler_decay(T, N)
fine_time = np.linspace(0, T, 400)
plt.figure(figsize=(8, 4))
plt.plot(fine_time, np.exp(-fine_time), "k--", label="Exact")
plt.plot(time, explicit, "o-", label="Explicit Euler")
plt.plot(time, implicit, "s-", label="Implicit Euler")
plt.xlabel("Time t")
plt.ylabel("x")
plt.title(f"N = {N}, h = {T / N:.3g}")
plt.legend()
plt.show()


**Answer 1.** Which method lies above the exact curve? What changes when you double $N$?

…


## 2. Find when explicit Euler fails

Run the four trials in **TO DO 2**. Fill the table below using the graphs.


In [ ]:
T = 10.0
# TO DO 2: run FOUR times: N = 20, 8, 5, 4. Keep T fixed.
# After each run, fill one row of the table below.
N = 20

# PLOTTING — supplied.
time, explicit, implicit = euler_decay(T, N)
fine_time = np.linspace(0, T, 400)
plt.figure(figsize=(8, 4))
plt.plot(fine_time, np.exp(-fine_time), "k--", label="Exact")
plt.plot(time, explicit, "o-", label="Explicit Euler")
plt.plot(time, implicit, "s-", label="Implicit Euler")
plt.axhline(0, color="grey", linewidth=0.8)
plt.xlabel("Time t")
plt.ylabel("x")
plt.title(f"h = {T / N:g}; explicit multiplier = {1 - T / N:g}")
plt.legend()
plt.show()


**Answer 2.** Describe signs and growth/decay.

| $N$ | $h$ | Explicit Euler | Implicit Euler |
|---|---|---|---|
| 20 | … | … | … |
| 8 | … | … | … |
| 5 | … | … | … |
| 4 | … | … | … |

What condition on $h>0$ makes explicit Euler decay in magnitude? Does a decaying numerical solution necessarily approximate the exact curve well?

…


## 3. Measure local and global error

Use **explicit Euler**. Local error compares one step from the exact starting value; global error compares the final value after all steps.

$$
L(h)=|x(h)-x_1|, \qquad G(T,h)=|x(T)-x_N|.
$$

Complete **TO DO 3.1–3.2** using `abs(...)`. Keep $T$ fixed.


In [ ]:
T = 2.0
step_counts = [5, 10, 20, 40, 80, 160]
step_sizes = []
local_errors = []
global_errors = []

for N in step_counts:
    h = T / N
    time, explicit, implicit = euler_decay(T, N)
    exact_one_step = np.exp(-h)
    exact_final = np.exp(-T)

    # TO DO 3.1: absolute difference between exact_one_step and explicit[1].
    local_error = None

    # TO DO 3.2: absolute difference between exact_final and explicit[-1].
    # [-1] means the last stored value.
    global_error = None

    # Supplied check and storage — do not edit.
    if local_error is None or global_error is None:
        raise ValueError("Complete TO DO 3.1 and 3.2, then rerun this cell.")
    step_sizes.append(h)
    local_errors.append(local_error)
    global_errors.append(global_error)


In [ ]:
# PLOTTING — run after the cell above; nothing to edit.
h_values = np.array(step_sizes)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].loglog(h_values, local_errors, "o-", label="Measured local error")
axes[0].loglog(h_values, local_errors[-1] * (h_values / h_values[-1])**2,
               "k:", label="Slope guide: h²")
axes[1].loglog(h_values, global_errors, "o-", label="Measured global error")
axes[1].loglog(h_values, global_errors[-1] * h_values / h_values[-1],
               "k:", label="Slope guide: h")
axes[0].set_title("One step")
axes[1].set_title(f"All steps to T = {T:g}")
for ax in axes:
    ax.set(xlabel="Step size h (log scale)", ylabel="Absolute error (log scale)")
    ax.legend()
plt.show()


**Answer 3.** Compare the slopes with the dotted guides. When $h$ is halved, local error is approximately divided by …; global error by … . Why must $T$ stay fixed?

…


## 4. Complete RK2 (Heun)

For $dx/dt=f(t,x)$, use

$$
k_1=f(t_n,x_n), \quad x^*=x_n+hk_1, \quad
k_2=f(t_{n+1},x^*), \quad x_{n+1}=x_n+\frac{h}{2}(k_1+k_2).
$$

`rhs` is $f$; `start_slope` is $k_1$; `end_slope` is $k_2$. Fill the two marked lines.


In [ ]:
def heun(rhs, T, N):
    h = T / N
    time = np.linspace(0, T, N + 1)
    x = np.zeros(N + 1)
    x[0] = 1.0

    for n in range(N):
        start_slope = rhs(time[n], x[n])

        # TO DO 4.1: calculate the prediction using x[n], h and start_slope.
        predicted_x = None
        if predicted_x is None:
            raise ValueError("Complete TO DO 4.1, then rerun both cells in Section 4.")

        end_slope = rhs(time[n + 1], predicted_x)

        # TO DO 4.2: advance x[n] using the average of the two slopes.
        next_x = None
        if next_x is None:
            raise ValueError("Complete TO DO 4.2, then rerun both cells in Section 4.")
        x[n + 1] = next_x

    return time, x


In [ ]:
def decay_rhs(t, x):
    return -x

T = 2.0
# TO DO 4.3: run once, then add 320 and 640 to this list and rerun.
step_counts = [5, 10, 20, 40, 80, 160]

# CALCULATION AND PLOTTING — supplied.
step_sizes = []
euler_errors = []
rk2_errors = []
for N in step_counts:
    time, explicit, implicit = euler_decay(T, N)
    time, rk2 = heun(decay_rhs, T, N)
    step_sizes.append(T / N)
    euler_errors.append(abs(explicit[-1] - np.exp(-T)))
    rk2_errors.append(abs(rk2[-1] - np.exp(-T)))

plt.figure(figsize=(8, 4))
plt.loglog(step_sizes, euler_errors, "o-", label="Explicit Euler")
plt.loglog(step_sizes, rk2_errors, "s-", label="RK2 (Heun)")
plt.xlabel("Step size h (log scale)")
plt.ylabel("Absolute error at T = 2 (log scale)")
plt.legend()
plt.show()

print("Error reduction for the last halving of h:")
print(f"Euler: {euler_errors[-2] / euler_errors[-1]:.3f}")
print(f"RK2:   {rk2_errors[-2] / rk2_errors[-1]:.3f}")


**Answer 4.** When $h$ is halved, RK2 error is approximately divided by … . Which method gives the smaller error for small $h$? Use $G\approx Ch^p$ to identify each method's order $p$.

…


## 5. Stop a circle becoming a spiral

$$
\frac{dx}{dt}=-y, \qquad \frac{dy}{dt}=x, \qquad (x(0),y(0))=(1,0).
$$

The exact trajectory is the unit circle; $E=x^2+y^2=1$.

**Symplectic Euler:** update $x$ first, then use the **new** $x$ to update $y$:

$$
x_{n+1}=x_n-hy_n, \qquad y_{n+1}=y_n+hx_{n+1}.
$$


In [ ]:
def rotation(T, N):
    h = T / N
    time = np.linspace(0, T, N + 1)
    x_euler = np.zeros(N + 1)
    y_euler = np.zeros(N + 1)
    x_symplectic = np.zeros(N + 1)
    y_symplectic = np.zeros(N + 1)
    x_euler[0] = 1.0
    x_symplectic[0] = 1.0

    for n in range(N):
        # Explicit Euler is supplied: both updates use OLD values.
        x_euler[n + 1] = x_euler[n] - h * y_euler[n]
        y_euler[n + 1] = y_euler[n] + h * x_euler[n]

        x_symplectic[n + 1] = x_symplectic[n] - h * y_symplectic[n]
        # TO DO 5.1: update y using y_symplectic[n], h and x_symplectic[n + 1].
        next_y = None
        if next_y is None:
            raise ValueError("Complete TO DO 5.1, then rerun this cell and the next.")
        y_symplectic[n + 1] = next_y

    return time, x_euler, y_euler, x_symplectic, y_symplectic


In [ ]:
# TO DO 5.2: try each pair. Each time, rerun this cell and the energy cell:
# (T, N) = (20, 100), then (100, 500), then (100, 1000).
T = 20.0
N = 100

# PLOTTING — supplied. Each panel has its own scale.
time, x_euler, y_euler, x_symplectic, y_symplectic = rotation(T, N)
angle = np.linspace(0, 2 * np.pi, 400)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
axes[0].plot(x_euler, y_euler, label="Explicit Euler")
axes[1].plot(x_symplectic, y_symplectic, label="Symplectic Euler")
for ax in axes:
    ax.plot(np.cos(angle), np.sin(angle), "k--", label="Exact circle")
    ax.set_aspect("equal", adjustable="box")
    ax.set(xlabel="x", ylabel="y")
    ax.legend(loc="upper right")
fig.suptitle(f"T = {T:g}, h = {T / N:g}")
plt.show()


For symplectic Euler, compare $E$ with the modified energy

$$
Q_h=x^2-hxy+y^2.
$$

Complete **TO DO 5.3**, then repeat the trials in 5.2.


In [ ]:
h = T / N
energy = x_symplectic**2 + y_symplectic**2

# TO DO 5.3: calculate Q_h using x_symplectic, y_symplectic and h.
# **2 squares every stored value.
modified_energy = None

if modified_energy is None:
    raise ValueError("Complete TO DO 5.3, then rerun this cell.")

# PLOTTING — supplied.
plt.figure(figsize=(8, 4))
plt.plot(time, energy, label="Ordinary energy E")
plt.plot(time, modified_energy, "k--", label="Modified energy Q_h")
plt.xlabel("Time t")
plt.ylabel("Energy")
plt.title(f"Symplectic Euler: h = {h:g}")
plt.legend()
plt.show()


**Answer 5.** Which trajectory spirals outwards? Which energy stays constant? What improves when $h$ is halved?

…

Keep $0<h<2$ here: symplectic Euler's trajectories need not stay bounded for larger steps.


## 6. If time allows: a time-dependent equation

$$
\frac{dx}{dt}=a\cos(t)x+b\sin(t), \qquad x(0)=1.
$$

Complete **TO DO 6.1–6.2** using

$$
x_{n+1}^{\mathrm{exp}}=x_n+h(a\cos(t_n)x_n+b\sin(t_n)),
$$

$$
x_{n+1}^{\mathrm{imp}}=\frac{x_n+hb\sin(t_{n+1})}{1-ha\cos(t_{n+1})}.
$$


In [ ]:
a = 0.5
b = 1.0
T = 5.0
# TO DO 6.3: after completing the formulas, run with N = 30, 60, 120.
N = 30
h = T / N
time = np.linspace(0, T, N + 1)
explicit = np.zeros(N + 1)
implicit = np.zeros(N + 1)
explicit[0] = 1.0
implicit[0] = 1.0

for n in range(N):
    start_time = time[n]
    end_time = time[n + 1]

    # TO DO 6.1: explicit update using explicit[n], start_time, a, b and h.
    # Use np.cos(...) and np.sin(...).
    next_explicit = None

    # TO DO 6.2: implicit update using implicit[n], end_time, a, b and h.
    next_implicit = None

    if next_explicit is None or next_implicit is None:
        raise ValueError("Complete TO DO 6.1 and 6.2, then rerun this cell.")
    explicit[n + 1] = next_explicit
    implicit[n + 1] = next_implicit


In [ ]:
# PLOTTING — rerun after each trial above; nothing to edit.
def changing_rhs(t, x):
    return a * np.cos(t) * x + b * np.sin(t)

time, rk2 = heun(changing_rhs, T, N)
plt.figure(figsize=(8, 4))
plt.plot(time, explicit, "o-", markersize=3, label="Explicit Euler")
plt.plot(time, implicit, "s-", markersize=3, label="Implicit Euler")
plt.plot(time, rk2, "^-", markersize=3, label="RK2 (Heun)")
plt.xlabel("Time t")
plt.ylabel("x")
plt.title(f"N = {N}, h = {h:.3g}")
plt.legend()
plt.show()


**Answer 6.** Which curve changes least as you halve $h$? Does agreement between numerical methods prove that they are exact?

…
